## Sample Notebook: Fast Fourier Transform

The main scope of this notebook is to do a deep study and understanding of the Fast Fourier Transform.\
\
What is a Fourier Transform?\
\
A Fourier Transform (FT) is a mathematical method that separates the components of a signal into its frequencies.\
It takes a function of a signal and through an integration it describes the different frequencies that make said signal.\
In a discrete sample, such as the ones used from CCDs or sensors, we need to use the Discrete form of the transform (DFT).\
But, in computational analysis, there is the Fast Fourier Transform (FFT), a highly efficient way to calculate the DFT.\
\
These transforms are highly susceptible to the variables in the sample.\
This notebook aims to visually experiment with this method.\
\
Variables:
 - signal to noise
 - sampling
 - frequency 1
 - frequency 2
 - amplitudes

In [ ]:
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np

Let's start by sampling a simple sine wave, and setting some variables that influence the behaviour

In [ ]:
def plot_signal(snr: float, dt: float, 
                    freq1: float, amp1: float, 
                    freq2: float, amp2: float, 
                    freq3: float, amp3: float, 
                    time: int):
    """
    This function takes the parameters needed
    to create and plot a simple sine wave.
    Interactive to showcase differences.

    Parameters
    ----------
    snr: float
    Signal to noise ratio of the signal.
    Adds perturbances to clean signal.

    dt: float
    Sampling time of the discrete signal.

    freq1, freq2, freq3: floats
    Dominant frequencies of the signal.

    amp1, amp2, amp3: floats
    Amplitude of dominant frequencies.

    time: integer
    Amount of time we register the signal.
    """
    fs = 1 / dt  # Sampling frequency (Hz)
    t = np.linspace(0, time, int(fs * time), endpoint=False)

    # Clean signal
    signal_clean = (
        amp1 * np.sin(2 * np.pi * freq1 * t)
        + amp2 * np.sin(2 * np.pi * freq2 * t)
        + (amp3 * np.sin(2 * np.pi * freq3 * t) if freq3 > 0 else 0)
    )

    # Add noise with given SNR
    power_signal = np.mean(signal_clean**2)
    power_noise = power_signal / (10 ** (snr / 10))
    noise = np.random.normal(scale=np.sqrt(power_noise), size=t.shape)
    signal = signal_clean + noise

    # Plot
    plt.figure(figsize=(15, 5))
    plt.plot(t, signal, color="c", alpha=0.7, label="Noisy signal")
    plt.plot(t, signal_clean, color="k", lw=1.2, label="Clean signal")
    plt.title(f"Signal (fs={fs:.2f} Hz, Nyquist={fs / 2:.2f} Hz)")
    plt.xlabel("Time [s]")
    plt.ylabel("Amplitude")
    plt.grid(True)
    plt.legend()
    plt.show()


# Interactive controls
widgets.interact(
    plot_signal,
    snr=widgets.FloatSlider(
        min=0, max=30, step=1, value=4, description="Signal/noise [dB]"
    ),
    dt=widgets.FloatSlider(
        min=0.001,
        max=0.05,
        step=0.001,
        value=0.025,
        description="Sampling time delta [s]",
    ),
    freq1=widgets.FloatSlider(
        min=1, max=50, step=0.01, value=5, description="Frequency 1 [Hz]"
    ),
    amp1=widgets.FloatSlider(
        min=1, max=10, step=0.1, value=2.5, description="Amplitude of 1 [Hz]"
    ),
    freq2=widgets.FloatSlider(
        min=1, max=50, step=0.1, value=19, description="Frequency 2 [Hz]"
    ),
    amp2=widgets.FloatSlider(
        min=1, max=10, step=0.1, value=2.5, description="Amplitude of 2 [Hz]"
    ),
    freq3=widgets.FloatSlider(
        min=0, max=50, step=0.01, value=0, description="Frequency 3 [Hz]"
    ),
    amp3=widgets.FloatSlider(
        min=1, max=10, step=0.1, value=2.5, description="Amplitude of 3 [Hz]"
    ),
    time=widgets.IntSlider(min=1, max=30, step=1, value=20, description="Duration [s]"),
)

Now, we'll do another signal, but we will also compute the FFT right away to see how they behave

In [ ]:
def real_fft_signal(snr: float, dt: float, 
                    freq1: float, amp1: float, 
                    freq2: float, amp2: float, 
                    freq3: float, amp3: float, 
                    time: int, real: bool):
    """
    Function to compute the Fast Fourier Transform
    of a sine wave signal created by the parameters.
    Interactive to showcase differences and what
    perturbes the result.

    Parameters
    ----------
    snr: float
    Signal to noise ratio of the signal.
    Adds perturbances to clean signal.

    dt: float
    Sampling time of the discrete signal.

    freq1, freq2, freq3: floats
    Dominant frequencies of the signal.

    amp1, amp2, amp3: floats
    Amplitude of dominant frequencies.

    time: integer
    Amount of time we register the signal.

    real: boolean
    Checkbox to see/not see the imaginary components.
    """
    fs = 1 / dt  # Sampling frequency (Hz)
    t = np.linspace(0, time, int(fs * time), endpoint=False)

    # Clean signal
    signal_clean = (
        amp1 * np.sin(2 * np.pi * freq1 * t)
        + amp2 * np.sin(2 * np.pi * freq2 * t)
        + (amp3 * np.sin(2 * np.pi * freq3 * t) if freq3 > 0 else 0)
    )

    # Add noise with given SNR
    power_signal = np.mean(signal_clean**2)
    power_noise = power_signal / (10 ** (snr / 10))
    noise = np.random.normal(scale=np.sqrt(power_noise), size=t.shape)
    signal = signal_clean + noise

    if real:
        vals = np.abs(np.fft.rfft(signal))
        freqs = np.fft.rfftfreq(len(signal), dt)
    else:
        vals = np.abs(np.fft.fft(signal))
        freqs = np.fft.fftfreq(len(signal), dt)

    fig, ax = plt.subplots(1, 2, sharex=False, figsize=(25, 10))

    ax[0].plot(t, signal, color="C6", alpha=0.7, label="Noisy signal")
    ax[0].plot(t, signal_clean, color="k", lw=1.2, label="Clean signal")
    ax[0].set_title(f"Signal with sampling = {fs} Hz ({dt} seconds)")
    ax[0].set_xlabel("Time [s]")
    ax[0].set_ylabel("Amplitude")
    ax[0].grid(True)
    ax[0].legend()

    ax[1].plot(freqs, vals, color="C6")
    ax[1].axvline(freq1, color="C2", linestyle=":", label=f"Frequency 1={freq1} Hz")
    ax[1].axvline(freq2, color="C2", linestyle=":", label=f"Frequency 2={freq2} Hz")
    ax[1].axvline(fs / 2, color="C4", linestyle="-")
    ax[1].set_title(f"FFT result, Nyquist={fs / 2} Hz")
    ax[1].set_xlabel("Frequency [Hz]")
    ax[1].set_ylabel("Amplitude")
    ax[1].grid(True)
    ax[1].legend()

    plt.show


# Interactive controls
widgets.interact(
    real_fft_signal,
    snr=widgets.FloatSlider(
        min=0, max=30, step=1, value=4, description="Signal/noise [dB]"
    ),
    dt=widgets.FloatSlider(
        min=0.001,
        max=0.05,
        step=0.001,
        value=0.025,
        description="Sampling time delta [s]",
    ),
    freq1=widgets.FloatSlider(
        min=1, max=50, step=0.01, value=5, description="Frequency 1 [Hz]"
    ),
    amp1=widgets.FloatSlider(
        min=1, max=10, step=0.1, value=2.5, description="Amplitude of 1 [Hz]"
    ),
    freq2=widgets.FloatSlider(
        min=1, max=50, step=0.1, value=19, description="Frequency 2 [Hz]"
    ),
    amp2=widgets.FloatSlider(
        min=1, max=10, step=0.1, value=2.5, description="Amplitude of 2 [Hz]"
    ),
    freq3=widgets.FloatSlider(
        min=0, max=50, step=0.01, value=0, description="Frequency 3 [Hz]"
    ),
    amp3=widgets.FloatSlider(
        min=1, max=10, step=0.1, value=2.5, description="Amplitude of 3 [Hz]"
    ),
    time=widgets.IntSlider(min=1, max=30, step=1, value=15, description="Duration [s]"),
    real=widgets.Checkbox(
        value=True, description="Only see real values?", disabled=False, indent=False
    ),
)

The Nyquist frequency is the half of the frequency of the sample and its the highest that can be accurately described in a FT analysis.\
If I have a sampling rate of 60Hz, the highest frequency I can detect with a Fourier Transform is around 30Hz.